# DeepLoc preprocessing

Input data for Figures 3A and 4A

In [ ]:
import os
from pathlib import Path
import urllib.request

import pandas as pd
import plotly.express as px

In [ ]:
_cwd = Path.cwd()
REPO_ROOT = next((str(p) for p in [_cwd, *_cwd.parents] if (p / ".git").exists()), str(_cwd))

RAW_FASTA    = os.path.join(REPO_ROOT, "examples/paper/data/deeploc/raw/deeploc_data.fasta")
OUTPUT_FASTA = os.path.join(REPO_ROOT, "examples/paper/data/deeploc/processed/deeploc_train.fasta")
OUTPUT_CSV   = os.path.join(REPO_ROOT, "examples/paper/data/deeploc/processed/deeploc_train_features.csv")

## 0. Download raw data

In [ ]:
DEEPLOC_URL = "https://services.healthtech.dtu.dk/services/DeepLoc-1.0/deeploc_data.fasta"

os.makedirs(os.path.dirname(RAW_FASTA), exist_ok=True)
if not os.path.exists(RAW_FASTA):
    print(f"Downloading {DEEPLOC_URL} ...")
    urllib.request.urlretrieve(DEEPLOC_URL, RAW_FASTA)
    print(f"saved to {RAW_FASTA}")
else:
    print(f"Already present: {RAW_FASTA}")

## 1. Parse FASTA headers

DeepLoc FASTA headers follow the format:
```
>ProteinName  SubcellularLocation-CellularLocation  [train|test]
```

For example: `>Q5I0E9 Cell.membrane-M` or `>Q9H400 Cell.membrane-M test`

In [ ]:
records = []   # list of dicts: {protein_name, subcellular_location, cellular_location, split, sequence}

with open(RAW_FASTA, "r") as fh:
    current_header = None
    current_seq = []
    for line in fh:
        line = line.rstrip()
        if line.startswith(">"):
            if current_header is not None:
                records.append({**current_header, "sequence": "".join(current_seq)})
            parts = line[1:].split()
            combined  = parts[1] if len(parts) > 1 else ""
            loc_parts = combined.split("-")
            current_header = {
                "protein_name":         parts[0],
                "subcellular_location":  loc_parts[0],
                "cellular_location":     loc_parts[1] if len(loc_parts) > 1 else "",
                "split":                parts[2] if len(parts) > 2 else "train",
            }
            current_seq = []
        else:
            current_seq.append(line)
    if current_header is not None:
        records.append({**current_header, "sequence": "".join(current_seq)})

df_all = pd.DataFrame(records)
df_all["sequence_length"] = df_all["sequence"].str.len()
print(f"Total records: {len(df_all)}")
print(df_all["split"].value_counts())

## 2. Filter training set

In [ ]:
df_train = df_all[df_all["split"] == "train"].reset_index(drop=True)
print(f"Training sequences: {len(df_train)}")

## 3. Save training FASTA

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_FASTA), exist_ok=True)
with open(OUTPUT_FASTA, "w") as fh:
    for _, row in df_train.iterrows():
        fh.write(f">{row['protein_name']}\n{row['sequence']}\n")
print(f"Saved {len(df_train)} training sequences to {OUTPUT_FASTA}")

## 4. Save features CSV

In [ ]:
features = df_train[["protein_name", "subcellular_location", "cellular_location", "sequence_length"]].copy()
features.to_csv(OUTPUT_CSV, index=False)
print(f"Saved features to {OUTPUT_CSV}")
features.head()

## 5. Class distribution

In [ ]:
counts = features["subcellular_location"].value_counts().reset_index()
counts.columns = ["subcellular_location", "count"]
print(counts.to_string(index=False))
fig = px.bar(
    counts.sort_values("count", ascending=False),
    x="subcellular_location",
    y="count",
    title="DeepLoc training set — class distribution",
    labels={"subcellular_location": "Subcellular localization", "count": "# sequences"},
    template="plotly_white",
)
fig.show()